In [4]:
"""
Preprocess and scale captured lab data (target domain). Reuse scaler from CICIDS2017,
and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.covariance import LedoitWolf

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [17]:
### Import and parse JSON files ###

# Creates a Path object pointing to the target-domain JSON directory.
data_dir = Path("data/raw/target")

# Read one target JSON file and return all flow records as a DataFrame.
# Expected format: {"flows": [{...}, {...}, ...]}
def read_target_json_flows(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if not isinstance(payload, dict):
        raise ValueError(
            f"Expected JSON object in {file_path}, got {type(payload).__name__}"
        )

    if "flows" not in payload:
        raise ValueError(f"Missing required key 'flows' in {file_path}")

    flows = payload["flows"]
    if not isinstance(flows, list):
        raise ValueError(
            f"Expected 'flows' to be a list in {file_path}, got {type(flows).__name__}"
        )

    if not all(isinstance(flow, dict) for flow in flows):
        raise ValueError(
            f"All entries in 'flows' must be JSON objects in {file_path}"
        )

    flow_df = pd.DataFrame.from_records(flows)
    expected_count = len(flows)

    if len(flow_df) != expected_count:
        raise ValueError(
            f"Flow count mismatch while loading {file_path}: "
            f"expected {expected_count}, loaded {len(flow_df)}"
        )

    return flow_df, expected_count

# Target domain spans multiple JSON exports; load and concatenate all of them.
json_files = sorted(data_dir.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {data_dir}")

per_file_counts = {}
dfs = []
total_expected_flows = 0

for file_path in json_files:
    flow_df, expected_count = read_target_json_flows(file_path)
    per_file_counts[file_path.name] = expected_count
    total_expected_flows += expected_count
    dfs.append(flow_df)

df = pd.concat(dfs, ignore_index=True, sort=False) if dfs else pd.DataFrame()

# Integrity check: ensure no flows were lost across concatenation.
if len(df) != total_expected_flows:
    raise ValueError(
        f"Total flow mismatch after concatenation: expected {total_expected_flows}, got {len(df)}"
    )

# Display a quick shape check and preview rows.
print(f"Loaded and concatenated {len(json_files)} target JSON files:")
for file_name in [p.name for p in json_files]:
    print(f"  - {file_name}: {per_file_counts[file_name]} flows")
print("Dataset shape:", df.shape)
print("Total flows loaded:", len(df))
print("Flow integrity check passed: no flows lost during ingestion.")
df.head()

Loaded and concatenated 8 target JSON files:
  - capture_20260414_185327.json: 12556 flows
  - capture_20260415_131451.json: 3311 flows
  - capture_20260415_133820.json: 45695 flows
  - capture_20260417_131850.json: 8397 flows
  - capture_20260417_142847.json: 9129 flows
  - capture_20260417_152940.json: 4678 flows
  - capture_20260419_094747.json: 49955 flows
  - capture_20260419_130742.json: 2172 flows
Dataset shape: (135893, 70)
Total flows loaded: 135893
Flow integrity check passed: no flows lost during ingestion.


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,-0.354687,-0.470908,-0.010425,-0.01095,-0.044950,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
1,-0.354687,-0.470914,-0.011684,-0.01095,-0.051373,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
2,-0.354687,-0.470868,-0.002872,-0.01095,-0.006417,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
3,-0.354687,-0.470868,-0.002872,-0.01095,-0.006417,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
4,-0.354687,-0.470908,-0.010425,-0.01095,-0.044950,-0.007566,-0.215804,0.802596,0.022986,-0.260452,...,-0.008911,0.002681,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764


In [18]:
### [Diagnostic] ###

# DataFrame shape snapshot at this stage of preprocessing.
print(f"DataFrame shape: {df.shape} (rows={len(df):,}, cols={len(df.columns):,})")

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

DataFrame shape: (135893, 70) (rows=135,893, cols=70)
Feature/column count: 70
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count

In [19]:
### Data sanitization ###

rows_before = len(df)

# Remove irrelevant columns.
df = df.drop(columns=["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"], errors="ignore")

# Remove duplicate flows.
rows_before_dedup = len(df)
df.drop_duplicates(inplace=True)
rows_removed_dedup = rows_before_dedup - len(df)
print(f"  (Deduplication: {rows_removed_dedup:,} rows removed, {len(df):,} rows remaining)")

# Remove leading/trailing spaces from all column names.
df.rename(columns=lambda x: x.strip(), inplace=True)

rows_after = len(df)
total_rows_removed = rows_before - rows_after
print(f"\nTotal rows removed during sanitization: {total_rows_removed:,} ({total_rows_removed/rows_before*100:.2f}%)")
print(f"Final row count: {rows_after:,}")

  (Deduplication: 106,933 rows removed, 28,960 rows remaining)

Total rows removed during sanitization: 106,933 (78.69%)
Final row count: 28,960


In [21]:
### [Diagnostic] ###

# DataFrame shape snapshot at this stage of preprocessing.
print(f"DataFrame shape: {df.shape} (rows={len(df):,}, cols={len(df.columns):,})")

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

DataFrame shape: (28960, 70) (rows=28,960, cols=70)
Feature/column count: 70
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Fwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
-

In [9]:
### Feature-space alignment ###
# (select features according to predetermined shared feature space)

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")
TARGET_FEATURE_MAP_PATH = Path("data/processed/target_feature_map.json")

# Handle missing files.
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )
if not TARGET_FEATURE_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target feature map not found at {TARGET_FEATURE_MAP_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features. Order must be preserved.
shared_features = load_feature_order(FEATURE_LIST_PATH)

with open(TARGET_FEATURE_MAP_PATH, "r", encoding="utf-8") as f:
    target_feature_map = json.load(f)
if not isinstance(target_feature_map, dict):
    raise ValueError(
        f"Expected JSON object at {TARGET_FEATURE_MAP_PATH}, got "
        f"{type(target_feature_map).__name__}"
    )

# Feature alignment uses canonical feature keys only; label handling stays separate.
feature_map = {k: v for k, v in target_feature_map.items() if k != "Label"}
missing_map_keys = sorted(set(shared_features) - set(feature_map))
extra_map_keys = sorted(set(feature_map) - set(shared_features))
if missing_map_keys or extra_map_keys:
    raise ValueError(
        f"Target feature map key mismatch. Missing keys: {missing_map_keys[:10]} "
        f"(total={len(missing_map_keys)}); extra keys: {extra_map_keys[:10]} "
        f"(total={len(extra_map_keys)}). Update {TARGET_FEATURE_MAP_PATH}."
    )

# Identify label column after sanitization trims any surrounding whitespace.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

mapped_label_col = target_feature_map.get("Label")
if mapped_label_col is not None:
    if not isinstance(mapped_label_col, str):
        raise ValueError(
            f"Label mapping in {TARGET_FEATURE_MAP_PATH} must be a string, "
            f"got {type(mapped_label_col).__name__}"
        )
    mapped_label_col = mapped_label_col.strip() or "Label"
    if mapped_label_col != label_col:
        raise ValueError(
            f"Label mapping mismatch: {TARGET_FEATURE_MAP_PATH} maps 'Label' to "
            f"'{mapped_label_col}', but dataset column is '{label_col}'."
        )

# Aligns a DataFrame to the canonical shared feature contract by renaming through the
# target feature map, dropping extra columns, checking optional fill behavior, and
# enforcing exact canonical order.
def align_feature_space(frame, feature_list, feature_map, fill_missing=False, fill_value=0.0):
    feature_list = list(feature_list)
    resolved_raw_columns = {}
    missing = []

    for canonical in feature_list:
        raw_name = feature_map.get(canonical, canonical)
        if not isinstance(raw_name, str):
            raise ValueError(
                f"Feature mapping for '{canonical}' must be a string, "
                f"got {type(raw_name).__name__}"
            )

        raw_name = raw_name.strip() or canonical
        resolved_raw_columns[canonical] = raw_name
        if raw_name not in frame.columns:
            missing.append((canonical, raw_name))

    # Strict mode (default): block pipeline if required mapped features are absent.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Mapped target features not found: {preview} (total={len(missing)})"
        )

    aligned_columns = {}
    for canonical in feature_list:
        raw_name = resolved_raw_columns[canonical]
        if raw_name in frame.columns:
            aligned_columns[canonical] = frame[raw_name]
        else:
            aligned_columns[canonical] = fill_value

    used_raw_columns = {
        raw_name for raw_name in resolved_raw_columns.values() if raw_name in frame.columns
    }
    extra = sorted(set(frame.columns) - used_raw_columns)

    aligned = pd.DataFrame(aligned_columns, index=frame.index).copy()
    return aligned, extra, missing

# Ensure index is contiguous before splitting/rejoining features and labels.
df = df.reset_index(drop=True)

# Align only feature columns; label handling happens separately.
feature_df = df.drop(columns=[label_col]).copy()
aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    feature_map,
    fill_missing=False,
    fill_value=0.0,
)

# Reattach labels by position (not index label) to avoid accidental NaNs.
labels_aligned = df[[label_col]].reset_index(drop=True)
aligned_X = aligned_X.reset_index(drop=True)
if len(aligned_X) != len(labels_aligned):
    raise ValueError(
        f"Feature/label row count mismatch after alignment: "
        f"X={len(aligned_X)}, y={len(labels_aligned)}"
    )
df = pd.concat([aligned_X, labels_aligned], axis=1)

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Loaded target feature map from {TARGET_FEATURE_MAP_PATH}")
print(f"Aligned feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")
print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")

Loaded shared feature list from data/processed/shared_feature_space.json
Loaded target feature map from data/processed/target_feature_map.json
Aligned feature count: 77
Dropped extra columns: 1
Missing required columns: 0
Missing labels after reattach: 0


In [10]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
TARGET_LABEL_MAP_PATH = Path("data/processed/target_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not TARGET_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target label map file not found at {TARGET_LABEL_MAP_PATH}"
    )

# Load the canonical label contract in a validated format.
with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_payload = json.load(f)

if isinstance(shared_label_payload, dict):
    if "labels" not in shared_label_payload:
        raise ValueError(
            f"Expected key 'labels' in {SHARED_LABEL_SPACE_PATH} when JSON object is provided."
        )
    shared_label_space = list(shared_label_payload["labels"])
elif isinstance(shared_label_payload, list):
    shared_label_space = list(shared_label_payload)
else:
    raise ValueError(
        f"Unsupported shared label space format in {SHARED_LABEL_SPACE_PATH}: "
        f"{type(shared_label_payload).__name__}"
    )

if not shared_label_space:
    raise ValueError(f"Shared label space in {SHARED_LABEL_SPACE_PATH} is empty")
if len(shared_label_space) != len(set(shared_label_space)):
    duplicate_labels = sorted(
        label for label in set(shared_label_space) if shared_label_space.count(label) > 1
    )
    raise ValueError(
        f"Duplicate canonical labels found in {SHARED_LABEL_SPACE_PATH}: {duplicate_labels}"
    )

with open(TARGET_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    target_label_map = json.load(f)
if not isinstance(target_label_map, dict):
    raise ValueError(
        f"Expected JSON object at {TARGET_LABEL_MAP_PATH}, got "
        f"{type(target_label_map).__name__}"
    )

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_canonical_labels = sorted(
    set(target_label_map.values()) - set(shared_label_space)
)
if invalid_canonical_labels:
    raise ValueError(
        "target_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_canonical_labels}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(target_label_map)

# Drop unmapped non-missing raw labels by design instead of crashing the pipeline.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
dropped_unmapped_count = int(unmapped_mask.sum())
if dropped_unmapped_count > 0:
    unmapped_raw = raw_labels[unmapped_mask]
    unmapped_counts = unmapped_raw.value_counts().sort_values(ascending=False)
    keep_mask = ~unmapped_mask

    print("Dropping unmapped raw labels from target dataset:")
    for raw_label, raw_count in unmapped_counts.items():
        print(f"- {raw_label}: {int(raw_count)}")

    df = df.loc[keep_mask].reset_index(drop=True)
    mapped_labels = mapped_labels.loc[keep_mask].reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
    mapped_labels = mapped_labels.reset_index(drop=True)

if df.empty:
    raise ValueError(
        "All rows were removed during target label alignment. "
        f"Check {TARGET_LABEL_MAP_PATH} and {SHARED_LABEL_SPACE_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Sanity check: print class-frequency table after label alignment.
label_counts = df[label_col].value_counts(dropna=False).sort_values(ascending=False)
label_freq = (label_counts / len(df) * 100).round(2)

print(f"Loaded shared label space from {SHARED_LABEL_SPACE_PATH}")
print(f"Loaded target label map from {TARGET_LABEL_MAP_PATH}")
print(f"Canonical label count: {len(shared_label_space)}")
print(f"Dropped unmapped rows: {dropped_unmapped_count}")
print(f"Rows retained after label alignment: {len(df)}")
print("Label category frequencies after alignment:")
for cls in label_counts.index:
    print(f"- {cls}: {int(label_counts[cls])} ({label_freq[cls]:.2f}%)")

Loaded shared label space from data/processed/shared_label_space.json
Loaded target label map from data/processed/target_label_map.json
Canonical label count: 13
Dropped unmapped rows: 0
Rows retained after label alignment: 2320849
Label category frequencies after alignment:
- Benign: 1717280 (73.99%)
- DDoS: 277806 (11.97%)
- Infiltration: 88527 (3.81%)
- DoS Hulk: 83665 (3.60%)
- Bot: 51142 (2.20%)
- SSH-Patator: 50143 (2.16%)
- DoS GoldenEye: 41406 (1.78%)
- DoS slowloris: 9908 (0.43%)
- Web Attack - Brute Force: 555 (0.02%)
- Web Attack - XSS: 228 (0.01%)
- Web Attack - Sql Injection: 84 (0.00%)
- DoS Slowhttptest: 55 (0.00%)
- FTP-Patator: 50 (0.00%)


In [11]:
### Train/Test Split ###
from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

X = df.drop(columns=[label_col]).copy()
y = df[label_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 1856679/464170


In [12]:
### Scaling (reuse scaler of source dataset) ###

SCALER_PATH = Path("models/source_scaler.joblib")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Source scaler not found at {SCALER_PATH}")

scaler = joblib.load(SCALER_PATH)

# Apply source-fitted scaler separately to target train/test split.
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve feature names/indexing.
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"Loaded scaler from {SCALER_PATH}")
print(f"Scaled splits shapes: train={X_train.shape}, test={X_test.shape}")

Loaded scaler from models/source_scaler.joblib
Scaled splits shapes: train=(1856679, 77), test=(464170, 77)


In [13]:
### Label encoding (reuse encoder of shared label space) ###

ENCODER_PATH = Path("models/label_encoder.joblib")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Label encoder not found at {ENCODER_PATH}")

le = joblib.load(ENCODER_PATH)

# Ensure train/test labels are fully compatible with source-fitted encoder.
raw_train_labels = y_train.astype("string").str.strip()
raw_test_labels = y_test.astype("string").str.strip()
all_unknown = sorted((set(raw_train_labels.dropna()) | set(raw_test_labels.dropna())) - set(le.classes_))
if all_unknown:
    raise ValueError(
        f"Found labels not present in fitted source encoder: {all_unknown[:10]} "
        f"(total={len(all_unknown)})."
    )

y_train = pd.Series(le.transform(raw_train_labels), index=y_train.index, name="Label")
y_test = pd.Series(le.transform(raw_test_labels), index=y_test.index, name="Label")

print(f"Loaded label encoder from {ENCODER_PATH}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Loaded label encoder from models/label_encoder.joblib
Encoded classes (13): ['Benign', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Infiltration', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']


In [ ]:
### Calculate and export covariance and mean statistics ###
# Note: calculated from target training split only.

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Improvement support: persist canonical feature ordering inside CORAL stats so
# evaluation can verify source/target schema parity before adaptation.
X_target_train_aligned = X_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_trg = X_target_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target training data.
X_trg_centered = X_trg - target_feature_mean

# Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# covariance estimate than plain sample covariance.
target_cov_estimator = LedoitWolf()
target_cov_estimator.fit(X_trg_centered)
target_covariance = np.asarray(target_cov_estimator.covariance_, dtype=np.float64)
target_covariance = (target_covariance + target_covariance.T) / 2.0

# Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
target_eigenvalues = np.linalg.eigvalsh(target_covariance)
target_min_eig = float(target_eigenvalues.min())
target_max_eig = float(target_eigenvalues.max())
target_cov_condition_number = float(np.linalg.cond(target_covariance))
target_covariance_ridge = 0.0

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
    "covariance_estimator": "LedoitWolf",
    "covariance_shrinkage": float(target_cov_estimator.shrinkage_),
    "min_eigenvalue_before_regularization": target_min_eig,
    "max_eigenvalue": target_max_eig,
    "covariance_condition_number": target_cov_condition_number,
    "covariance_ridge": target_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print("Verified: CORAL stats were computed from target train split only.")
print(f"Train rows used for CORAL: {len(X_target_train_aligned)}")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")
print(f"Covariance estimator: LedoitWolf (shrinkage={target_cov_estimator.shrinkage_:.6f})")
print(f"Covariance eigenvalues: min={target_min_eig:.6e}, max={target_max_eig:.6e}")
print(f"Covariance condition number: {target_cov_condition_number:.6e}")

CORAL target statistics extracted and saved successfully.
Verified: CORAL stats were computed from target train split only.
Train rows used for CORAL: 1856679
Saved to: models/coral_target_stats.joblib
Features: 77
Covariance shape: (77, 77)
Covariance estimator: LedoitWolf (shrinkage=1.000000)
Covariance eigenvalues: min=6.406657e+02, max=6.406657e+02
Covariance condition number: 1.000000e+00


In [15]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled target dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: inspect every unique raw value stored in the 'Label' feature.
label_feature = "Label"
if label_feature not in df.columns:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Expected '{label_feature}' column in compiled target dataset but it was not found. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique 'Label' values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 78
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
- CWE Flag Count
- EC

In [16]:
### Export processed data ###

# Create output directory for processed target train/test splits.
output_dir = Path("data/processed/target")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed target train data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "train.csv", index=False)

# Save processed target test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(train_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 1856679 samples
  Test: 464170 samples
  Output directory: data/processed/target
